## Data Collection — 15-Year Indian Market Data
Downloads OHLCV data for macro assets + 10 high-influence Indian blue-chip stocks via `yfinance`.

In [1]:
import pandas as pd
import yfinance as yf
from functools import reduce
from datetime import date

In [2]:
# Tickers: macro proxies + 10 high-influence NSE blue-chips
asset_tickers = {
    # Broad Markets & Macro
    'nifty50'    : '^NSEI',
    'sp500'      : '^GSPC',
    'usd_inr'    : 'INR=X',
    'gold'       : 'GC=F',
    'brent'      : 'BZ=F',

    # High-Influence Indian Stocks (NSE)
    'reliance'   : 'RELIANCE.NS',
    'hdfcbank'   : 'HDFCBANK.NS',
    'tcs'        : 'TCS.NS',
    'infy'       : 'INFY.NS',
    'icicibank'  : 'ICICIBANK.NS',
    'lt'         : 'LT.NS',
    'itc'        : 'ITC.NS',
    'sunpharma'  : 'SUNPHARMA.NS',
    'sbin'       : 'SBIN.NS',
    'bhartiartl' : 'BHARTIARTL.NS',
}

In [3]:
START_DATE = '2010-01-01'
END_DATE   = str(date.today())   # always fetches up to today

dataframes = []
failed     = []

for prefix, ticker in asset_tickers.items():
    print(f'Downloading {prefix} ({ticker})...')

    df = yf.Ticker(ticker).history(start=START_DATE, end=END_DATE)

    if df.empty:
        print(f'  WARNING: No data for {ticker} — skipping.')
        failed.append(prefix)
        continue

    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.columns = [f'{prefix}_{c.lower()}' for c in df.columns]
    df.index   = df.index.tz_localize(None)   # strip timezone
    df.index.name = 'date'
    dataframes.append(df)

print(f'\nDownloaded: {len(dataframes)} assets | Failed: {failed}')


Downloaded: 15 assets | Failed: []


In [4]:
# Outer-join on date so no trading day is lost
final_df = reduce(
    lambda l, r: pd.merge(l, r, on='date', how='outer'),
    dataframes
)
final_df = final_df.sort_index()

OUTPUT_FILE = 'master_market_data_2010_2026.csv'
final_df.to_csv(OUTPUT_FILE)

print(f'Saved → {OUTPUT_FILE}  |  Shape: {final_df.shape}')
display(final_df.tail(3))

Saved → master_market_data_2010_2026.csv  |  Shape: (4253, 75)


,nifty50_open,nifty50_high,nifty50_low,nifty50_close,nifty50_volume,sp500_open,sp500_high,sp500_low,sp500_close,sp500_volume,...,sbin_open,sbin_high,sbin_low,sbin_close,sbin_volume,bhartiartl_open,bhartiartl_high,bhartiartl_low,bhartiartl_close,bhartiartl_volume
date,,,,,,,,,,,,,,,,,,,,,
2026-04-16,24385.199219,24400.949219,24102.800781,24196.750000,508000.0,7037.779785,7051.229980,7008.520020,7041.279785,5.173650e+09,...,1078.000000,1084.150024,1062.000000,1067.150024,20313165.0,1863.500000,1864.900024,1823.199951,1840.599976,15871247.0
2026-04-17,24165.900391,24371.900391,24096.050781,24353.550781,498600.0,7074.549805,7147.520020,7074.549805,7126.060059,6.145300e+09,...,1067.000000,1082.150024,1060.000000,1080.250000,16823879.0,1837.099976,1851.400024,1829.699951,1846.900024,7344787.0
2026-04-20,24391.500000,24480.650391,24241.250000,24364.849609,415900.0,7117.049805,7122.649902,7084.410156,7109.140137,4.661130e+09,...,1080.300049,1120.949951,1075.849976,1107.849976,31179584.0,1865.000000,1865.000000,1839.199951,1846.099976,9243641.0
